<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="locally_deployed_nim.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a href="locally_deployed_nim.ipynb">2</a>
        <a>3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <!-- <span style="float: left; width: 33%; text-align: right;"><a href="challenge.ipynb">Next Notebook</a></span> -->
</div>

# LoRAアダプタによるNVIDIA NIMの実行
---

**概要:** このラボの目的は、LoRAアダプターを中心に、PEFT(Parameter Efficient Fine Tuning)の概念を紹介することです。


## Low Rank Adapters(LoRA)

大規模なLLMの完全なフルファインチューニング（つまりモデルの全パラメータを更新すること）は、モデル全体の学習に必要な計算インフラ量のために困難な場合があります。 インフラストラクチャーのコストは、ユーザーが複数の大きなモデルをメモリにホストするか、モデル全体がスワップイン・アウトされることによる待ち時間の増加を許容する必要があり、デプロイ時にも増加します。したがって、最小限の変更回数で目的を達成し、重みの読み取りと書き込みを行う必要があります。

一般的な方法は、最先端の[Parameter-Efficient Finetuning (PEFT)](https://github.com/huggingface/peft/tree/main) アプローチを使用することです。[PEFT](https://arxiv.org/abs/2305.16742)は、モデルの全パラメータではなく、少数の（余分な）モデルパラメータをファインチューニングすることを可能にし、計算コストとストレージコストを大幅に削減します。PEFTを実装する方法の1つは、Low-Rank Adaptation (LoRA) テクニックを採用することです。LoRA手法では、ダウンストリームのタスクで学習可能なパラメータの数を大幅に減らすことで、ファインチューニングをより効率的に行うことが出来ます。これは、事前に訓練されたモデルの重みを凍結し、Transformerアーキテクチャの各層に訓練可能なランク分解行列を注入することで実現されます。[LoRAの作者](https://arxiv.org/abs/2106.09685)によると、学習可能なパラメータの数を10k倍削減するほか、GPUの消費量を3倍削減し、推論レイテンシなしで高いスループットを実現します。LoRAの簡単な背景については、こちらの[リンク](https://huggingface.co/docs/peft/main/en/conceptual_guides/lora)を参照してください。

<center><img src="imgs/lora-arch.png" height="500px" width="900px"  /></center>
<center>  LoRA reparametrization and Weight merging. <a href="https://huggingface.co/docs/peft/main/en/conceptual_guides/lora"> View source</a> </center>

## NVIDIA NIMのLoRAサポート

LoRAの多面的なメリットを活かすため、NIMではベースモデルの上位に複数のLoRAインスタンスを配置することができます。これにより、下図の例のように、同じバックボーンを使いながら、複数のジャンルのユーザーアプリケーションにカスタムメイドで対応することができます：

<center><img src="imgs/nvidia-nim-dynamic-lora-architecture.png" height="500px" width="900px"  /></center>
<center>  LoRA reparametrization and Weight merging. <a href="https://huggingface.co/docs/peft/main/en/conceptual_guides/lora"> View source</a> </center>


## モデルのプロファイル

NVIDIA NIMは、計算リソースから引き出せる最高のパフォーマンスを提供するように設計されています。これは、利用可能なハードウェア（すなわち、アーキテクチャやGPUの数）、要件などに基づいて特別に調整された最適化されたモデルプロファイルによって行われます。完全なリストは[こちら](https://docs.nvidia.com/nim/large-language-models/latest/support-matrix.html#supported-lora-formats)にあります。

先にダウンロードした`Llama3-8b` NIMで利用可能なモデルプロファイルをリストアップしてみましょう。その前に、ポートを設定する必要があります。

In [ ]:
import random
import socket
import os
def find_available_port(start=11000, end=11999):
    while True:
        # ポートをランダムに選択
        port = random.randint(start, end)
        
        # ソケットを作成し、ポートにバインド
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # バインドが成功した場合、ポートは空きの状態であるため、ポートを返す
                return port
            except OSError:
                # バインドに失敗した場合、ポートは使用中であるため、次のイテレーションに継続
                continue

# 利用可能なポートを見つけて出力
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

In [ ]:
!docker run -it --rm --gpus all --shm-size=16GB   -u $(id -u) -p $CONTAINER_PORT:8000  nvcr.io/nim/meta/llama3-8b-instruct:1.0.3 list-model-profiles

**想定される出力:**

```text

===========================================
== NVIDIA Inference Microservice LLM NIM ==
===========================================

NVIDIA Inference Microservice LLM NIM Version 1.0.3
Model: meta/llama3-8b-instruct

Container image Copyright (c) 2016-2024, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This NIM container is governed by the NVIDIA AI Product Agreement here:
https://www.nvidia.com/en-us/data-center/products/nvidia-ai-enterprise/eula/.
A copy of this license can be found under /opt/nim/LICENSE.

The use of this model is governed by the AI Foundation Models Community License
here: https://docs.nvidia.com/ai-foundation-models-community-license.pdf.

ADDITIONAL INFORMATION: Meta Llama 3 Community License, Built with Meta Llama 3.
A copy of the Llama 3 license can be found under /opt/nim/MODEL_LICENSE.

SYSTEM INFO
- Free GPUs:
  -  [20b2:10de] (0) NVIDIA A100-SXM4-80GB (A100 80GB) [current utilization: 1%]
MODEL PROFILES
- Compatible with system and runnable:
  - 8835c31752fbc67ef658b20a9f78e056914fdef0660206d82f252d62fd96064d (vllm-fp16-tp1)
  - With LoRA support:
    - 8d3824f766182a754159e88ad5a0bd465b1b4cf69ecf80bd6d6833753e945740 (vllm-fp16-tp1-lora)
- Incompatible with system:
  - dcd85d5e877e954f26c4a7248cd3b98c489fbde5f1cf68b4af11d665fa55778e (tensorrt_llm-h100-fp8-tp2-latency)
  - f59d52b0715ee1ecf01e6759dea23655b93ed26b12e57126d9ec43b397ea2b87 (tensorrt_llm-l40s-fp8-tp2-latency)
  - 30b562864b5b1e3b236f7b6d6a0998efbed491e4917323d04590f715aa9897dc (tensorrt_llm-h100-fp8-tp1-throughput)
  - 09e2f8e68f78ce94bf79d15b40a21333cea5d09dbe01ede63f6c957f4fcfab7b (tensorrt_llm-l40s-fp8-tp1-throughput)
  - a93a1a6b72643f2b2ee5e80ef25904f4d3f942a87f8d32da9e617eeccfaae04c (tensorrt_llm-a100-fp16-tp2-latency)
  - e0f4a47844733eb57f9f9c3566432acb8d20482a1d06ec1c0d71ece448e21086 (tensorrt_llm-a10g-fp16-tp2-latency)
  - 879b05541189ce8f6323656b25b7dff1930faca2abe552431848e62b7e767080 (tensorrt_llm-h100-fp16-tp2-latency)
  - 24199f79a562b187c52e644489177b6a4eae0c9fdad6f7d0a8cb3677f5b1bc89 (tensorrt_llm-l40s-fp16-tp2-latency)
...
```
<br/>

サンプルのプロファイルを取り上げてみましょう: `cce57ae50c3af15625c1668d5ac4ccbe82f40fa2e8379cc7b842cc6c976fd334 (tensorrt_llm-a100-fp16-tp1-throughput-lora)`

この説明文はNIMsモデルプロファイルの詳細を示しています:
- `tensorrt_llm` 推論エンジン
- Nvidia `A100` GPU用
- `fp16`精度
- 単一の `テンソル並列` (つまり単一デバイス)
- `throughput` 重視で
- `lora`をサポート

### モデルプロファイルと共にNVIDIA NIMを動かす

In [3]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

In [4]:
from os.path import expanduser
home = expanduser("~")

In [ ]:
import os
try:
    os.mkdir(f"{home}/.cache/nim/")
    print("Cache Created")
except:
    print("Cache Exists..")
    pass
    

In [6]:
os.environ['LOCAL_NIM_CACHE'] = f"{home}/.cache/nim/"

リストアップされたプロファイルでNVIDIA NIMを走らせてみましょう。

In [ ]:
! docker run -it --rm --gpus all --shm-size=16GB \
-e NIM_MODEL_PROFILE=8d3824f766182a754159e88ad5a0bd465b1b4cf69ecf80bd6d6833753e945740 \
-e NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) -p $CONTAINER_PORT:8000  nvcr.io/nim/meta/llama3-8b-instruct:1.0.3

**想定される出力:**

```

===========================================
== NVIDIA Inference Microservice LLM NIM ==
===========================================

NVIDIA Inference Microservice LLM NIM Version 1.0.3
Model: nim/meta/llama3-8b-instruct

Container image Copyright (c) 2016-2024, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This NIM container is governed by the NVIDIA AI Product Agreement here:
https://www.nvidia.com/en-us/data-center/products/nvidia-ai-enterprise/eula/.
A copy of this license can be found under /opt/nim/LICENSE.

The use of this model is governed by the AI Foundation Models Community License
here: https://docs.nvidia.com/ai-foundation-models-community-license.pdf.

ADDITIONAL INFORMATION: Meta Llama 3 Community License, Built with Meta Llama 3. 
A copy of the Llama 3 license can be found under /opt/nim/MODEL_LICENSE.

2024-08-14 07:27:01,272 [INFO] PyTorch version 2.2.2 available.
2024-08-14 07:27:02,170 [WARNING] [TRT-LLM] [W] Logger level already set from environment. Discard new verbosity: error
2024-08-14 07:27:02,171 [INFO] [TRT-LLM] [I] Starting TensorRT-LLM init.
2024-08-14 07:27:02,214 [INFO] [TRT-LLM] [I] TensorRT-LLM inited.
[TensorRT-LLM] TensorRT-LLM version: 0.10.1.dev2024053000
INFO 08-14 07:27:03.173 api_server.py:489] NIM LLM API version 1.0.0
INFO 08-14 07:27:03.175 ngc_profile.py:217] Running NIM without LoRA. Only looking for compatible profiles that do not support LoRA.
INFO 08-14 07:27:03.175 ngc_profile.py:219] Detected 0 compatible profile(s).
INFO 08-14 07:27:03.175 ngc_profile.py:221] Detected additional 2 compatible profile(s) that are currently not runnable due to low free GPU memory.
ERROR 08-14 07:27:03.176 utils.py:21] You are running NIM without LoRA, but the selected profile '8d3824f766182a754159e88ad5a0bd465b1b4cf69ecf80bd6d6833753e945740' has LoRA enabled. Please select a profile that does not have LoRA enabled, or alternatively, run NIM with LoRA enabled.
```

**NIMのログには、LoRAが有効なモデル・プロファイルを選択したが、LoRAアダプタの定義を渡さなかったことが明確に示されています。**

### LoRA セットアップ概要

環境変数で定義されたコンフィギュレーションを使って、NIMを拡張してLoRAモデルを提供することができます。ただし、条件があります：  使用する基礎となるNVIDIA NIMは、LoRAのベースモデルと互換性がなければなりません。したがって、コンフィギュレーションが必要です。詳細は後述します。

#### LoRA アダプタ

- LoRAアダプタを `NGC` または `Hugging Face` からダウンロードするか、`カスタムLoRAアダプタ` を使用します。
- LoRAアダプタは別々のディレクトリに格納する必要があり、`LOCAL_PEFT_DIRECTORY`ディレクトリの中に1つ以上のLoRAディレクトリが必要です。
- ロードされた LoRA アダプタの名前は、アダプタのディレクトリ名と一致していなければなりません。 

NIM for LLMはNeMoとHugging Face Transformersの互換フォーマットをサポートしています。

- NeMoフォーマットのLoRAディレクトリには、拡張子が`.nemo`のファイルが1つ含まれていなければなりません。`.nemo`ファイルの名前は親ディレクトリの名前と一致する必要はありません。サポートされるターゲットモジュールは`["gate_proj", "o_proj", "up_proj", "down_proj", "k_proj", "q_proj", "v_proj", "attention_qkv"]`です。

- Hugging Face Transformersで学習されたLoRAアダプタがサポートされます。LoRAは、adapter_config.jsonファイルと{adapter_model.safetensors, adapter_model.bin}ファイルのいずれかを含む必要があります。NVIDIA NIMでサポートされるターゲットモジュールは`["gate_proj", "o_proj", "up_proj", "down_proj", "k_proj", "q_proj", "v_proj"]`です。

#### LoRA モデルのディレクトリ構造

例えば、1つ以上のLoRA（`LOCAL_PEFT_DIRECTORY`）を格納するディレクトリは以下のように構成します： 
- `lolas`には、dockerコンテナに渡すディレクトリ名を `LOCAL_PEFT_DIRECTORY` の値として指定します。
- そして、ロードされる LoRA は `llama3-8b-math`、`llama3-8b-math-hf`、`llama3-8b-squad`、`llama3-8b-squad-hf` という名前になります。


```text

loras
├── llama3-8b-math
│   └── llama3_8b_math.nemo
├── llama3-8b-math-hf
│   ├── adapter_config.json
│   └── adapter_model.bin
├── llama3-8b-squad
│   └── squad.nemo
└── llama3-8b-squad-hf
    ├── adapter_config.json
    └── adapter_model.safetensors

```

### LoRAアダプタのダウンロード

前述したように、NVIDIAモデルレジストリNGCまたはHuggingFaceから、事前に訓練されたアダプタをダウンロードすることができます。 LoRAモデルの重みは、特定のベースモデルに結びついていることに注意してください。NVIDIA NIMによって提供されるものと同じモデルでチューニングされたLoRAモデルのみをデプロイする必要があります。HuggingFaceからアダプターをダウンロードする方法の例を以下に示します。 

*huggingface-cli ログインが必須であることに注意してください。*

```text
#export and make directory
export LOCAL_PEFT_DIRECTORY=~/loras
mkdir $LOCAL_PEFT_DIRECTORY

#download a LoRA from Hugging Face Hub
mkdir $LOCAL_PEFT_DIRECTORY/llama3-lora
huggingface-cli download <Hugging Face LoRA name> adapter_config.json adapter_model.safetensors --local-dir $LOCAL_PEFT_DIRECTORY/llama3-lora

#create permissions
chmod -R 777 $LOCAL_PEFT_DIRECTORY

```


次に、NGCから`llama3-8b-instruct`用のLoRAアダプターをダウンロードする方法を示します。

### NVCR (NVIDIA Container Registry)へのログイン

NIM の docker イメージにアクセスするには、`docker login nvcr.io`でログインする必要があります。このプロセスには、デフォルトのユーザー名 `--username $oauthtoken` と `--password-stdin` が必要で、`$NGC_API_KEY` の値を受け付けます。

In [ ]:
! echo -e "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

In [9]:
# LoRA用のディレクトリを作成
!mkdir -p loras

In [10]:
# vLLM-formatのLoRAをダウンロード
# NOTE: LoRAのダウンロードはNGCで行う場合、configのセットアップが必要です。BootcampではLoRAが事前にダウンロードされています。
# !ngc registry model download-version "nim/meta/llama3-8b-instruct-lora:hf-math-v1" --dest "./loras/" 

**想定される出力:** (NGCからダウンロードする場合)

```text
Downloaded 37.14 MB in 2s, Download speed: 18.57 MB/s               
--------------------------------------------------------------------------------
   Transfer id: llama3-8b-instruct-lora_vhf-math-v1
   Download status: Completed
   Downloaded local path: /home/<USER>/NIMs_Bootcamp/loras/llama3-8b-instruct-lora_vhf-math-v1-1
   Total files downloaded: 4
   Total downloaded size: 37.14 MB
   Started at: 2024-08-15 12:37:20.924505
   Completed at: 2024-08-15 12:37:22.928174
   Duration taken: 2s
   
...

```

### NIMへのLoRAアダプタの追加

docker からアクセスできるように、LoRA ファイルにすべての表示権限を付与します。

In [11]:
! chmod -R 777 loras/
os.environ['NIM_PEFT_SOURCE'] = f'{os.getcwd()}/loras'
os.environ['LOCAL_PEFT_DIRECTORY'] = f'{os.getcwd()}/loras'

上述したように、loraのコンフィギュレーションはファイルとフォルダーの構造に敏感なので、正しいデプロイメントを確実にするために、.ipynbチェックポイントに入り込んでいる可能性があるものを取り除きます。

In [12]:
! find . -type d -name ".ipynb_checkpoints" -exec rm -rf {} +


NVIDIA NIM dockerコンテナを実行します。

In [ ]:
os.environ['LORA_NIM_DOCKER']="nim_llm_with_lora"
! docker run -it -d --rm --gpus all --shm-size=16GB --name=$LORA_NIM_DOCKER   -e NGC_API_KEY  -e NIM_PEFT_SOURCE -v $LOCAL_PEFT_DIRECTORY:$NIM_PEFT_SOURCE -v $LOCAL_NIM_CACHE:/opt/nim/.cache  -u $(id -u) -p $CONTAINER_PORT:8000 nvcr.io/nim/meta/llama3-8b-instruct:1.0.3

# ローカルのNIMコンテナが完全にロードされ、保留状態にならないように、一定時間待ちます。
! sleep 60

dockerログでNVIDIA NIMが動作していることを確認します。

In [ ]:
! docker logs --tail 45 $LORA_NIM_DOCKER

**想定される出力**:  

```
INFO 09-10 12:17:48.149 api_server.py:456] Serving endpoints:
  0.0.0.0:8000/openapi.json
  0.0.0.0:8000/docs
  0.0.0.0:8000/docs/oauth2-redirect
  0.0.0.0:8000/metrics
  0.0.0.0:8000/v1/health/ready
  0.0.0.0:8000/v1/health/live
  0.0.0.0:8000/v1/models
  0.0.0.0:8000/v1/version
  0.0.0.0:8000/v1/chat/completions
  0.0.0.0:8000/v1/completions
INFO 09-10 12:17:48.149 api_server.py:460] An example cURL request:
curl -X 'POST' \
  'http://0.0.0.0:8000/v1/chat/completions' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
    "model": "meta/llama3-8b-instruct",
    "messages": [
      {
        "role":"user",
        "content":"Hello! How are you?"
      },
      {
        "role":"assistant",
        "content":"Hi! I am quite well, how can I help you today?"
      },
      {
        "role":"user",
        "content":"Can you write me a song?"
      }
    ],
    "top_p": 1,
    "n": 1,
    "max_tokens": 15,
    "stream": true,
    "frequency_penalty": 1.0,
    "stop": ["hello"]
  }'

INFO 09-10 12:17:48.196 server.py:82] Started server process [33]
INFO 09-10 12:17:48.197 on.py:48] Waiting for application startup.
INFO 09-10 12:17:48.224 on.py:62] Application startup complete.
INFO 09-10 12:17:48.225 server.py:214] Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

### 推論の実行

NIMsのtriton推論サーバーで指定された`/models`エンドポイントから、利用可能なモデルエンドポイントとホストされたLoRAを素早くチェックすることができます。

In [ ]:
! curl -s http://0.0.0.0:${CONTAINER_PORT}/v1/models | jq

**想定される出力:**

```json

{
  "object": "list",
  "data": [
    {
      "id": "meta/llama3-8b-instruct",
      "object": "model",
      "created": 1723751020,
      "owned_by": "system",
      "root": "meta/llama3-8b-instruct",
      "parent": null,
      "permission": [
        {
          "id": "modelperm-0f38baa00db14455846b31342c8b4367",
          "object": "model_permission",
          "created": 1723751020,
          "allow_create_engine": false,
          "allow_sampling": true,
          "allow_logprobs": true,
          "allow_search_indices": false,
          "allow_view": true,
          "allow_fine_tuning": false,
          "organization": "*",
          "group": null,
          "is_blocking": false
        }
      ]
    },
    {
      "id": "llama3-8b-instruct-lora_vhf-math-v1",
      "object": "model",
      "created": 1723751020,
      "owned_by": "system",
      "root": "meta/llama3-8b-instruct",
      "parent": null,
      "permission": [     
    ...   
  ]
}

```

簡単な推論リクエストを実行します。

In [16]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

model = "llama3-8b-instruct-lora_vhf-math-v1"
llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),model=model)

In [ ]:
out = llm.invoke("7+9は？")

print(out)

**想定される出力:**
```python
AIMessage(content='7 + 9 = 16', response_metadata={'role': 'assistant', 'content': '7 + 9 = 16', 'token_usage': {'prompt_tokens': 17, 'total_tokens': 25, 'completion_tokens': 8}, 'finish_reason': 'stop', 'model_name': 'llama3-8b-instruct-lora_vhf-math-v1'}, id='run-c52c2e00-80d4-409b-88c1-8b977b6e9041-0', role='assistant')
```

In [ ]:
out = llm.invoke("$(3,7)$と$(5,1)$の中点の座標を求めよ。")

print(out.content)

実行中のdockerコンテナを停止します。

In [ ]:
!docker container stop $LORA_NIM_DOCKER 

### カスタムLoRAアダプタの作成

カスタムLoRAアダプターを使用するには、<a href="custom_lora.ipynb"> custom_lora notebook </a>を使用して構築する必要があります。プロセスが完了したら、以下のセルを実行して、カスタムLoRAアダプタをNIMでデプロイしてください。 最初のステップは、アダプタを `loras` ディレクトリにコピーすることです。

In [22]:
!cp  -r model/Llama-3-8b-instruct-hf-finetune loras 

パーミッションの作成

In [23]:
!chmod -R 777 loras/

NIM dockerコンテナを実行します。

In [ ]:
! docker run -it --rm -d --gpus all --shm-size=16GB   -e NGC_API_KEY  -e NIM_PEFT_SOURCE -v $LOCAL_PEFT_DIRECTORY:$NIM_PEFT_SOURCE -v $LOCAL_NIM_CACHE:/opt/nim/.cache  -u $(id -u) -p $CONTAINER_PORT:8000  nvcr.io/nim/meta/llama3-8b-instruct:1.0.0


# ! sleep 60 # wait for container to load fully

In [ ]:
! docker logs --tail 50 $(docker ps -q -l) # view the last logs from the container

### カスタムLoRAアダプタによる推論の実行

NIMsのtriton推論サーバ経由で指定された`/models`エンドポイントから、利用可能なモデルエンドポイントとホストされたLoRAを素早くチェックすることができます。

In [ ]:
!curl -s http://0.0.0.0:${CONTAINER_PORT}/v1/models | jq

モデルが到達可能で、稼働していることを確認するテストを行います。

In [ ]:
!curl -X 'POST' \
    "http://0.0.0.0:${CONTAINER_PORT}/v1/completions" \
    -H 'accept: application/json' \
    -H 'Content-Type: application/json' \
    -d '{"model": "Llama-3-8b-instruct-hf-finetune", \
         "prompt": "昔々あるところに",\
         "max_tokens": 64        }'


簡単な推論リクエストを実行します。

In [28]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

model = "Llama-3-8b-instruct-hf-finetune" # カスタムLoRAアダプタを使用

llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),model=model, temperature=0.7, max_new_tokens=256, top_p=0.4)

In [ ]:
output = llm.invoke("仙台を舞台にした短い物語を書いて")
print(output.content)

実行中のdockerコンテナを停止します。

In [ ]:
!docker container stop $LORA_NIM_DOCKER 

---

## 参照

- https://docs.nvidia.com/nim/large-language-models/latest/peft.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<br>
<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="locally_deployed_nim.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a href="locally_deployed_nim.ipynb">2</a>
        <a>3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <!-- <span style="float: left; width: 33%; text-align: right;"><a href="challenge.ipynb">Next Notebook</a></span> -->
</div>

<br>
<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>